# 02 — Whisper Audio and Precision Diagnostic

The October 22 pilot completed, but three predictions repeated text until the token limit. This notebook investigates the setup before scaling up.

**One controlled question:** does changing model/input computation from FP16 to FP32 alter these failures? Repeat both language modes at both precisions in the same session, with the same model revision, SDPA attention implementation, token limit, generation settings, and exact five validation waveforms. FP32 is a diagnostic, not a promised fix. No repetition penalties, new prompts, or changed references are introduced.

The original pilot manifest and settings are embedded below as data; you do not need to upload its ZIP again. Original results remain unchanged. Download the new ZIP for comparison.

In a **fresh Colab GPU session**, run step 1.1, restart the session, then resume at **1.2**. Listen to all five clips and record the requested checks before running inference. This is a setup diagnostic, not a CMI analysis, timing benchmark, or completed reference audit.


### 1.1 Install the Same Pilot Dependencies

Keep the original versions to avoid confounding precision with a library upgrade. The known pip conflicts with preinstalled Gradio/Diffusers/gcsfs remain; this notebook does not use them. Restart after installation and resume at 1.2.

**Restart is required after installing or changing packages**, even if pip does not request it. Save your notebook first, including any completed `reviews` list. Choose **Runtime → Restart session**, then rerun from section 1.2 onward without rerunning installation. Restarting clears variables; saved cell edits remain. Mixing old imported modules with newly installed files can cause missing Hugging Face attributes.


In [ ]:
%pip install -q "transformers==4.57.1" "huggingface-hub==0.36.2" "datasets==4.4.1" "librosa==0.11.0" "soundfile==0.13.1" "jiwer==4.0.0"


### 1.2 Import Libraries and Check GPU

Use one GPU for both precision runs. Models are loaded separately to limit memory use. The original FP16 pilot is a historical baseline; the rerun in this session is the direct comparison.


In [ ]:
import copy
import hashlib
import io
import json
import math
import platform
import subprocess
import sys
import unicodedata
import zipfile
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import huggingface_hub
from huggingface_hub import constants as hub_constants

# Package changes need a session restart to clear already imported modules.
assert version('huggingface-hub') == huggingface_hub.__version__ == '0.36.2', (
    'Run section 1.1, restart the Colab session, then resume at section 1.2.'
)
assert hasattr(hub_constants, 'HF_HUB_ENABLE_HF_TRANSFER'), (
    'Mixed Hugging Face modules are still loaded. Restart the Colab session before continuing.'
)
assert version('transformers') == '4.57.1', 'Run section 1.1, restart, then resume at 1.2.'

import numpy as np
import soundfile as sf
import torch
from huggingface_hub import snapshot_download
from datasets import load_dataset
from IPython.display import Audio, display
from jiwer import cer, wer
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor


import gc
from transformers import GenerationConfig

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Colab GPU runtime before continuing.')
DEVICE = 'cuda:0'
torch.manual_seed(42)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print({name: version(name) for name in ['transformers', 'datasets', 'huggingface-hub', 'fsspec', 'soundfile', 'jiwer']})


### 1.3 Load the Original Manifest and Settings

These are recorded data from your supplied ZIP. Do not change clip identities or decoding settings during this diagnostic.


In [ ]:
baseline = json.loads('{"created_at_utc": "2026-10-22T08:49:26.480010+00:00", "dataset": {"id": "saileshbro/nepali-cs-asr", "revision": "dd00ca41dd38774cd6097c372c16939d7f3746df", "config": "default", "split": "validation"}, "model": {"id": "openai/whisper-large-v3-turbo", "revision": "41f01f3fe87f28c78e2fbf8b568835947dd65ed9"}, "selection": {"speech_type": "cs", "seed": 42, "count": 5, "shuffle_buffer": 1000}, "manifest": [{"transcription": "होइन धेरै जसो student के हुन्छन् तपाईँको study एकदम hard गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_devanagari": "होइन धेरै जसो स्टुडेन्ट के हुन्छन् तपाईँको सटुडय एकदम हारड गर्छन् मतलब दिन रात 1 गर्छन् उनीहरू", "transcription_romanized": "hoina dherai jaso student ke hunchan tapaiko study ekdam hard garchan matlab dina rata 1 garchan uniharu", "cmi": 18.75, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "JF_HlxEXcHQ", "start_ms": 90896, "end_ms": 99488, "duration_s": 8.592, "clip_id": "validation/JF_HlxEXcHQ/90896-99488", "decoded_duration_s": 8.592, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "5d7728fbbec30b6f4f42110ecd48a76d19dea181673460552219a1f9beefcaa8"}, {"transcription": "यसलाई easily यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त so half life को meaning के हो आधा हुने हो नि त so तिम्रो like", "transcription_devanagari": "यसलाई ईसिलय यतिकै पनि निकाल्न मिल्ने थियो 25% भनेको हो नि त सो हाफ लाइफ को meaning के हो आधा हुने हो नि त सो तिम्रो लाइक", "transcription_romanized": "yaslai easily yatikai pani nikalna milne thiyo 25% bhaneko ho ni ta so half life ko meaning ke ho aadha hune ho ni ta so timro like", "cmi": 26.92, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 633264, "end_ms": 642192, "duration_s": 8.928, "clip_id": "validation/zJYS-XNDP2Q/633264-642192", "decoded_duration_s": 8.928, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "a19babdc7287647007c9675646769c74d8dd186b780fb5ce9ae8a74f887c52ef"}, {"transcription": "यो हामीले निकाल्ने हो नि त mass भनेको basically N N हाम्रो कति निक्लियो यति निक्लियो into molar mass molar mass हाम्रो कति हुन्छ भन्दाखेरि", "transcription_devanagari": "यो हामीले निकाल्ने हो नि त म्यास भनेको बेसिकली एन एन हाम्रो कति निक्लियो यति निक्लियो इन्टु molar म्यास molar म्यास हाम्रो कति हुन्छ भन्दाखेरि", "transcription_romanized": "yo hamile nikalne ho ni ta mass bhaneko basically N N hamro kati nikliyo yati nikliyo into molar mass molar mass hamro kati huncha bhandakheri", "cmi": 36.0, "speech_type": "cs", "speaker_id": "UCeq9rTo9W93NfJNP9n1eRGw", "video_id": "zJYS-XNDP2Q", "start_ms": 1512240, "end_ms": 1525520, "duration_s": 13.28, "clip_id": "validation/zJYS-XNDP2Q/1512240-1525520", "decoded_duration_s": 13.28, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "574ae80ba5da885329f71a80a96040083e9b54e656a9f3f7f9aa15b8980c94b4"}, {"transcription": "physical interpretation यदि गर्न सक्यो भने होइन त्यसबाट हामी question from the predictable last year को question बाट गर्न सक्छौँ marks ल्याउन सजिलो भइदिन्छ repetition rate होइन", "transcription_devanagari": "फिजिकल इन्टरप्रिटेसन यदि गर्न सक्यो भने होइन त्यसबाट हामी क्वेश्चन फ्रम द प्रेडिक्टेबल लास्ट यीयर को क्वेश्चन बाट गर्न सक्छौँ मार्कस ल्याउन सजिलो भइदिन्छ रेपिटिशन रेट होइन", "transcription_romanized": "physical interpretation yadi garna sakyo bhane hoina tyasbata hami question from the predictable last year ko question bata garna sakchau marks lyauna sajilo bhaidincha repetition rate hoina", "cmi": 44.44, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1205232, "end_ms": 1216352, "duration_s": 11.12, "clip_id": "validation/TN8IzjVNnzs/1205232-1216352", "decoded_duration_s": 11.12, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "aae11f0b20a48ac8d2b029fe82fbc8b5cda3bc675594c692368a1ba2d30de2d9"}, {"transcription": "Electric charge field potential capacitor DC circuit chapter हरू छन् होइन जसमध्ये mostly तपाईँको concept", "transcription_devanagari": "इलेक्ट्रिक चार्ज फिल्ड पोटेनशल क्यापासिटर डीसी सर्किट च्याप्टर हरू छन् होइन जसमध्ये मोसटलय तपाईँको कन्सेप्ट", "transcription_romanized": "Electric charge field potential capacitor DC circuit chapter haru chan hoina jasamadhye mostly tapaiko concept", "cmi": 33.33, "speech_type": "cs", "speaker_id": "UCBbxIYJ35vGP8qWMJn3RmTA", "video_id": "TN8IzjVNnzs", "start_ms": 1169296, "end_ms": 1178352, "duration_s": 9.056, "clip_id": "validation/TN8IzjVNnzs/1169296-1178352", "decoded_duration_s": 9.056, "sample_rate": 16000, "source_channels": 1, "audio_sha256": "52e59651d4a7af08a78092d441cebc8414550ca2721ceb5476644d8d1d8162a1"}], "summary": {"auto": {"clips": 5, "reference_words": 111, "wer": 2.8018018018018016, "cer": 1.8628571428571428, "outputs_without_eos": 1}, "forced_ne": {"clips": 5, "reference_words": 111, "wer": 1.117117117117117, "cer": 1.5257142857142858, "outputs_without_eos": 2}}, "generation_configs": {"auto": {"max_length": 448, "max_new_tokens": 440, "min_length": 0, "min_new_tokens": null, "early_stopping": false, "max_time": null, "stop_strings": null, "do_sample": false, "num_beams": 1, "use_cache": true, "cache_implementation": null, "cache_config": null, "return_legacy_cache": null, "prefill_chunk_size": null, "temperature": 1.0, "top_k": 50, "top_p": 1.0, "min_p": null, "typical_p": 1.0, "epsilon_cutoff": 0.0, "eta_cutoff": 0.0, "repetition_penalty": 1.0, "encoder_repetition_penalty": 1.0, "length_penalty": 1.0, "no_repeat_ngram_size": 0, "bad_words_ids": null, "renormalize_logits": false, "forced_bos_token_id": null, "forced_eos_token_id": null, "remove_invalid_values": false, "exponential_decay_length_penalty": null, "suppress_tokens": [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50359, 50360, 50361, 50362, 50363], "begin_suppress_tokens": [220, 50257], "sequence_bias": null, "token_healing": false, "guidance_scale": null, "watermarking_config": null, "num_return_sequences": 1, "output_attentions": false, "output_hidden_states": false, "output_scores": false, "output_logits": null, "return_dict_in_generate": false, "pad_token_id": 50257, "bos_token_id": 50257, "eos_token_id": 50257, "encoder_no_repeat_ngram_size": 0, "decoder_start_token_id": 50258, "is_assistant": false, "num_assistant_tokens": 20, "num_assistant_tokens_schedule": "constant", "assistant_confidence_threshold": 0.4, "prompt_lookup_num_tokens": null, "max_matching_ngram_size": null, "assistant_early_exit": null, "assistant_lookbehind": 10, "target_lookbehind": 10, "disable_compile": false, "low_memory": null, "penalty_alpha": null, "dola_layers": null, "diversity_penalty": 0.0, "num_beam_groups": 1, "constraints": null, "force_words_ids": null, "_from_model_config": false, "transformers_version": "4.57.1", "alignment_heads": [[2, 4], [2, 11], [3, 3], [3, 6], [3, 11], [3, 14]], "forced_decoder_ids": null, "is_multilingual": true, "lang_to_id": {"<|af|>": 50327, "<|am|>": 50334, "<|ar|>": 50272, "<|as|>": 50350, "<|az|>": 50304, "<|ba|>": 50355, "<|be|>": 50330, "<|bg|>": 50292, "<|bn|>": 50302, "<|bo|>": 50347, "<|br|>": 50309, "<|bs|>": 50315, "<|ca|>": 50270, "<|cs|>": 50283, "<|cy|>": 50297, "<|da|>": 50285, "<|de|>": 50261, "<|el|>": 50281, "<|en|>": 50259, "<|es|>": 50262, "<|et|>": 50307, "<|eu|>": 50310, "<|fa|>": 50300, "<|fi|>": 50277, "<|fo|>": 50338, "<|fr|>": 50265, "<|gl|>": 50319, "<|gu|>": 50333, "<|haw|>": 50352, "<|ha|>": 50354, "<|he|>": 50279, "<|hi|>": 50276, "<|hr|>": 50291, "<|ht|>": 50339, "<|hu|>": 50286, "<|hy|>": 50312, "<|id|>": 50275, "<|is|>": 50311, "<|it|>": 50274, "<|ja|>": 50266, "<|jw|>": 50356, "<|ka|>": 50329, "<|kk|>": 50316, "<|km|>": 50323, "<|kn|>": 50306, "<|ko|>": 50264, "<|la|>": 50294, "<|lb|>": 50345, "<|ln|>": 50353, "<|lo|>": 50336, "<|lt|>": 50293, "<|lv|>": 50301, "<|mg|>": 50349, "<|mi|>": 50295, "<|mk|>": 50308, "<|ml|>": 50296, "<|mn|>": 50314, "<|mr|>": 50320, "<|ms|>": 50282, "<|mt|>": 50343, "<|my|>": 50346, "<|ne|>": 50313, "<|nl|>": 50271, "<|nn|>": 50342, "<|no|>": 50288, "<|oc|>": 50328, "<|pa|>": 50321, "<|pl|>": 50269, "<|ps|>": 50340, "<|pt|>": 50267, "<|ro|>": 50284, "<|ru|>": 50263, "<|sa|>": 50344, "<|sd|>": 50332, "<|si|>": 50322, "<|sk|>": 50298, "<|sl|>": 50305, "<|sn|>": 50324, "<|so|>": 50326, "<|sq|>": 50317, "<|sr|>": 50303, "<|su|>": 50357, "<|sv|>": 50273, "<|sw|>": 50318, "<|ta|>": 50287, "<|te|>": 50299, "<|tg|>": 50331, "<|th|>": 50289, "<|tk|>": 50341, "<|tl|>": 50348, "<|tr|>": 50268, "<|tt|>": 50351, "<|uk|>": 50280, "<|ur|>": 50290, "<|uz|>": 50337, "<|vi|>": 50278, "<|yi|>": 50335, "<|yo|>": 50325, "<|yue|>": 50358, "<|zh|>": 50260}, "max_initial_timestamp_index": 50, "no_timestamps_token_id": 50364, "prev_sot_token_id": 50362, "return_timestamps": false, "task_to_id": {"transcribe": 50360, "translate": 50359}, "language": null, "task": "transcribe"}, "forced_ne": {"max_length": 448, "max_new_tokens": 440, "min_length": 0, "min_new_tokens": null, "early_stopping": false, "max_time": null, "stop_strings": null, "do_sample": false, "num_beams": 1, "use_cache": true, "cache_implementation": null, "cache_config": null, "return_legacy_cache": null, "prefill_chunk_size": null, "temperature": 1.0, "top_k": 50, "top_p": 1.0, "min_p": null, "typical_p": 1.0, "epsilon_cutoff": 0.0, "eta_cutoff": 0.0, "repetition_penalty": 1.0, "encoder_repetition_penalty": 1.0, "length_penalty": 1.0, "no_repeat_ngram_size": 0, "bad_words_ids": null, "renormalize_logits": false, "forced_bos_token_id": null, "forced_eos_token_id": null, "remove_invalid_values": false, "exponential_decay_length_penalty": null, "suppress_tokens": [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50359, 50360, 50361, 50362, 50363], "begin_suppress_tokens": [220, 50257], "sequence_bias": null, "token_healing": false, "guidance_scale": null, "watermarking_config": null, "num_return_sequences": 1, "output_attentions": false, "output_hidden_states": false, "output_scores": false, "output_logits": null, "return_dict_in_generate": false, "pad_token_id": 50257, "bos_token_id": 50257, "eos_token_id": 50257, "encoder_no_repeat_ngram_size": 0, "decoder_start_token_id": 50258, "is_assistant": false, "num_assistant_tokens": 20, "num_assistant_tokens_schedule": "constant", "assistant_confidence_threshold": 0.4, "prompt_lookup_num_tokens": null, "max_matching_ngram_size": null, "assistant_early_exit": null, "assistant_lookbehind": 10, "target_lookbehind": 10, "disable_compile": false, "low_memory": null, "penalty_alpha": null, "dola_layers": null, "diversity_penalty": 0.0, "num_beam_groups": 1, "constraints": null, "force_words_ids": null, "_from_model_config": false, "transformers_version": "4.57.1", "alignment_heads": [[2, 4], [2, 11], [3, 3], [3, 6], [3, 11], [3, 14]], "forced_decoder_ids": null, "is_multilingual": true, "lang_to_id": {"<|af|>": 50327, "<|am|>": 50334, "<|ar|>": 50272, "<|as|>": 50350, "<|az|>": 50304, "<|ba|>": 50355, "<|be|>": 50330, "<|bg|>": 50292, "<|bn|>": 50302, "<|bo|>": 50347, "<|br|>": 50309, "<|bs|>": 50315, "<|ca|>": 50270, "<|cs|>": 50283, "<|cy|>": 50297, "<|da|>": 50285, "<|de|>": 50261, "<|el|>": 50281, "<|en|>": 50259, "<|es|>": 50262, "<|et|>": 50307, "<|eu|>": 50310, "<|fa|>": 50300, "<|fi|>": 50277, "<|fo|>": 50338, "<|fr|>": 50265, "<|gl|>": 50319, "<|gu|>": 50333, "<|haw|>": 50352, "<|ha|>": 50354, "<|he|>": 50279, "<|hi|>": 50276, "<|hr|>": 50291, "<|ht|>": 50339, "<|hu|>": 50286, "<|hy|>": 50312, "<|id|>": 50275, "<|is|>": 50311, "<|it|>": 50274, "<|ja|>": 50266, "<|jw|>": 50356, "<|ka|>": 50329, "<|kk|>": 50316, "<|km|>": 50323, "<|kn|>": 50306, "<|ko|>": 50264, "<|la|>": 50294, "<|lb|>": 50345, "<|ln|>": 50353, "<|lo|>": 50336, "<|lt|>": 50293, "<|lv|>": 50301, "<|mg|>": 50349, "<|mi|>": 50295, "<|mk|>": 50308, "<|ml|>": 50296, "<|mn|>": 50314, "<|mr|>": 50320, "<|ms|>": 50282, "<|mt|>": 50343, "<|my|>": 50346, "<|ne|>": 50313, "<|nl|>": 50271, "<|nn|>": 50342, "<|no|>": 50288, "<|oc|>": 50328, "<|pa|>": 50321, "<|pl|>": 50269, "<|ps|>": 50340, "<|pt|>": 50267, "<|ro|>": 50284, "<|ru|>": 50263, "<|sa|>": 50344, "<|sd|>": 50332, "<|si|>": 50322, "<|sk|>": 50298, "<|sl|>": 50305, "<|sn|>": 50324, "<|so|>": 50326, "<|sq|>": 50317, "<|sr|>": 50303, "<|su|>": 50357, "<|sv|>": 50273, "<|sw|>": 50318, "<|ta|>": 50287, "<|te|>": 50299, "<|tg|>": 50331, "<|th|>": 50289, "<|tk|>": 50341, "<|tl|>": 50348, "<|tr|>": 50268, "<|tt|>": 50351, "<|uk|>": 50280, "<|ur|>": 50290, "<|uz|>": 50337, "<|vi|>": 50278, "<|yi|>": 50335, "<|yo|>": 50325, "<|yue|>": 50358, "<|zh|>": 50260}, "max_initial_timestamp_index": 50, "no_timestamps_token_id": 50364, "prev_sot_token_id": 50362, "return_timestamps": false, "task_to_id": {"transcribe": 50360, "translate": 50359}, "language": "ne", "task": "transcribe"}}, "generation_overrides": {"return_timestamps": false, "temperature": 0.0, "return_dict_in_generate": true}, "runtime": {"python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]", "platform": "Linux-6.6.122+-x86_64-with-glibc2.39", "torch": "2.11.0+cu130", "cuda": "13.0", "gpu": "Tesla T4", "dtype": "torch.float16"}, "source_archive": "whisper_pilot_20261022T084926480010Z.zip"}')

DATASET_ID = baseline['dataset']['id']
DATASET_REVISION = baseline['dataset']['revision']
SPLIT = baseline['dataset']['split']
MODEL_ID = baseline['model']['id']
MODEL_REVISION = baseline['model']['revision']
assert SPLIT == 'validation'
expected_clips = {row['clip_id']: row for row in baseline['manifest']}
if len(expected_clips) != 5:
    raise ValueError('Expected the five unique clips from the original pilot.')
print('Original pilot:', baseline['source_archive'])
print('Original scores:', json.dumps(baseline['summary'], indent=2))


### 2.1 Reuse the Same Scoring and Audio Checks

The same canonical references, normalization, decoder, and checks are used in both notebooks.


In [ ]:
columns = ['audio', 'transcription', 'transcription_devanagari', 'transcription_romanized', 'cmi', 'speech_type', 'speaker_id', 'video_id', 'start_ms', 'end_ms', 'duration_s']

def normalize_text(text):
    text = unicodedata.normalize('NFC', text).lower()
    text = text.translate(str.maketrans('०१२३४५६७८९', '0123456789'))
    text = ''.join(' ' if unicodedata.category(char).startswith('P') else char for char in text)
    return ' '.join(text.split())

assert normalize_text('Hello,  SCHOOL!') == 'hello school'
assert normalize_text('नेपालमा १२ school छन्।') == 'नेपालमा 12 school छन्'
assert normalize_text('cafe\u0301') == normalize_text('café')
assert normalize_text('school') != normalize_text('स्कुल')
assert normalize_text(' ' + chr(9) + chr(10)) == ''
assert wer(['a b', 'c d'], ['a', 'c d e']) == 0.5
assert wer('a b', '') == 1.0
assert wer('a', 'a b c') == 2.0
assert cer('ab', 'a') == 0.5
print('Normalization and scoring self-checks passed.')

def prepare_clip(row):
    video_id = row.get('video_id')
    start, end = row.get('start_ms'), row.get('end_ms')
    if (
        not isinstance(video_id, str) or not video_id.strip()
        or any(not isinstance(value, int) or isinstance(value, bool) for value in (start, end))
        or not 0 <= start < end
    ):
        raise ValueError('Missing or invalid video_id/start_ms/end_ms; inspect clip provenance.')
    key = f'{SPLIT}/{video_id}/{start}-{end}'
    reference, cmi = row.get('transcription'), row.get('cmi')
    if not isinstance(reference, str) or not normalize_text(reference):
        raise ValueError(f'Empty or invalid reference for {key}.')
    if isinstance(cmi, bool) or not isinstance(cmi, (int, float)) or not math.isfinite(cmi) or not 0 <= cmi <= 50:
        raise ValueError(f'Invalid bilingual CMI for {key}: {cmi}.')
    encoded = row.get('audio')
    if not isinstance(encoded, dict) or not isinstance(encoded.get('bytes'), bytes) or not encoded['bytes']:
        raise ValueError(f'Expected embedded WAV bytes for {key}; inspect the dataset format.')
    waveform, sample_rate = sf.read(io.BytesIO(encoded['bytes']), dtype='float32', always_2d=True)
    if sample_rate != 16000 or waveform.shape[0] == 0 or waveform.shape[1] == 0 or not np.isfinite(waveform).all():
        raise ValueError(f'Invalid or non-finite audio for {key}; expected nonempty 16 kHz audio.')
    channels = waveform.shape[1]
    audio = waveform.mean(axis=1)
    duration = len(audio) / sample_rate
    expected = row.get('duration_s')
    if (
        not 0 < duration <= 30
        or isinstance(expected, bool) or not isinstance(expected, (int, float)) or not math.isfinite(expected)
        or abs(duration - expected) > 0.02
        or abs(duration - (end - start) / 1000) > 0.02
    ):
        raise ValueError(f'Audio duration does not match metadata/provenance for {key}.')
    identity = {name: row[name] for name in columns if name != 'audio'}
    identity.update(
        clip_id=key, decoded_duration_s=duration, sample_rate=sample_rate, source_channels=channels,
        audio_sha256=hashlib.sha256(audio.tobytes()).hexdigest(),
    )
    return {'identity': identity, 'audio': audio}


def inspect_clip(clip):
    identity, audio = clip['identity'], clip['audio']
    print('CLIP:', identity['clip_id'])
    print('REFERENCE:', identity['transcription'])
    print('CMI:', identity['cmi'], '| SPEECH TYPE:', identity['speech_type'])
    print('Waveform shape:', audio.shape, '| dtype:', audio.dtype, '| type:', type(audio))
    print('Source channels:', identity['source_channels'], '| Sample rate:', identity['sample_rate'])
    print('Dataset duration:', identity['duration_s'], '| Decoded duration:', identity['decoded_duration_s'])
    print('Min:', float(audio.min()), '| Max:', float(audio.max()))
    print('Mean absolute amplitude:', float(np.abs(audio).mean()))
    display(Audio(audio, rate=identity['sample_rate']))


### 2.2 Recover Exactly the Original Five Waveforms

Stream the pinned validation split without shuffling. Retain only matching provenance tuples, reject ambiguous matches, and verify every stored metadata field and decoded audio hash against the original manifest. A mismatch stops the run. No fresh samples are substituted.


In [ ]:
stream = load_dataset(DATASET_ID, name='default', split=SPLIT, revision=DATASET_REVISION, streaming=True)
stream = stream.decode(False).select_columns(columns)
found = {}
for row in stream:
    key = f'{SPLIT}/{row["video_id"]}/{row["start_ms"]}-{row["end_ms"]}'
    if key not in expected_clips:
        continue
    if key in found:
        raise ValueError(f'Ambiguous duplicate provenance: {key}.')
    clip = prepare_clip(row)
    expected = expected_clips[key]
    if clip['identity'] != expected:
        raise ValueError(f'Audio hash or metadata changed for {key}; do not compare different inputs.')
    found[key] = clip
missing = set(expected_clips) - set(found)
if missing:
    raise ValueError(f'Missing original clips: {sorted(missing)}.')
clips = [found[row['clip_id']] for row in baseline['manifest']]
print('Verified five exact audio hashes and all recorded clip metadata.')


### 2.3 Listen to All Five Clips

Run this cell to display all five audio players and references. There are no interactive input prompts; the cell finishes after displaying the clips. Listen and compare the spoken content with each reference, then record your checks in section 2.4. Script/spelling differences alone do not necessarily mean different spoken content.


In [ ]:
for number, clip in enumerate(clips, start=1):
    print(f'\nCLIP {number}')
    inspect_clip(clip)
print('All five clips displayed. Listen, then fill in section 2.4 below.')


### 2.4 Record Your Listening Checks

Edit each pending value below to match, mismatch, or unsure after listening to that clip; add notes in the second string if helpful. Use unsure when you cannot judge the audio or a technical term. Pending means not yet reviewed and stops the cell with a clear message. These notes do not alter the scoring references or remove clips from the diagnostic. Rerunning resets the exported checks to the values you entered here.


In [ ]:
reviews = [
    ('pending', ''),  # Clip 1: replace pending after listening
    ('pending', ''),  # Clip 2
    ('pending', ''),  # Clip 3
    ('pending', ''),  # Clip 4
    ('pending', ''),  # Clip 5
]
listening_checks = []
if len(reviews) != len(clips):
    raise ValueError('Enter exactly one review for each of the five clips.')
for number, (clip, (verdict, note)) in enumerate(zip(clips, reviews), start=1):
    if verdict not in {'match', 'mismatch', 'unsure'}:
        raise ValueError(f'Clip {number} still needs review. Replace pending with match, mismatch, or unsure.')
    if not isinstance(note, str):
        raise ValueError(f'Clip {number}: notes must be text.')
    listening_checks.append({'clip_id': clip['identity']['clip_id'], 'reference_check': verdict, 'notes': note})
print('Saved all five listening checks. Continue to section 3.1.')


### 3.1 Rerun FP16 and FP32 with Identical Decoding

Run 20 transcriptions: five clips × two language settings × two precisions. Reload weights for each precision, so FP32 starts from the checkpoint rather than rounded FP16 weights. No quantization or automatic mixed precision is used. An FP16 rerun controls for changes since the original Colab session.

Do not interpret a missing EOS token as proof of hallucination. All full outputs remain in the export; the console prints only a short preview. Stop and share the error if the GPU runs out of memory.

Processor files are downloaded from the same pinned revision and loaded locally to avoid an optional additional_chat_templates lookup. Model weights, decoding settings, and clip identities are unchanged.


In [ ]:
if len(listening_checks) != len(clips):
    raise ValueError('Complete the listening checks before inference.')
# Load locally so tokenizer setup does not query Whisper's absent chat-template folder.
processor_dir = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    allow_patterns=[
        'config.json', 'preprocessor_config.json', 'tokenizer*.json',
        'vocab.json', 'merges.txt', 'normalizer.json',
        'added_tokens.json', 'special_tokens_map.json',
    ],
)
processor = AutoProcessor.from_pretrained(processor_dir, local_files_only=True)
predictions = []
mode_configs = {}
for precision, DTYPE in [('fp16', torch.float16), ('fp32', torch.float32)]:
    gc.collect()
    torch.cuda.empty_cache()
    torch.manual_seed(42)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE, use_safetensors=True, attn_implementation='sdpa',
    ).to(DEVICE).eval()
    id_to_language = {token_id: token for token, token_id in model.generation_config.lang_to_id.items()}
    for mode, language in [('auto', None), ('forced_ne', 'ne')]:
        config = GenerationConfig.from_dict(baseline['generation_configs'][mode])
        config.language = language
        config.task = 'transcribe'
        config.forced_decoder_ids = None
        config.do_sample = False
        config.num_beams = 1
        config.max_new_tokens = baseline['generation_configs'][mode]['max_new_tokens']
        mode_configs[precision + '/' + mode] = config.to_dict()
        for clip in clips:
            inputs = processor(clip['audio'], sampling_rate=16000, return_tensors='pt', return_attention_mask=True)
            inputs = inputs.to(DEVICE, dtype=DTYPE)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, generation_config=copy.deepcopy(config),
                    return_timestamps=False, temperature=0.0, return_dict_in_generate=True
                )
            token_ids = generated.sequences[0].detach().cpu().tolist()
            hypothesis = processor.batch_decode(generated.sequences, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
            reference = clip['identity']['transcription']
            normalized_reference = normalize_text(reference)
            normalized_hypothesis = normalize_text(hypothesis)
            record = {
                **clip['identity'], 'mode': mode, 'precision': precision, 'hypothesis': hypothesis,
                'normalized_reference': normalized_reference, 'normalized_hypothesis': normalized_hypothesis,
                'wer': wer(normalized_reference, normalized_hypothesis),
                'cer': cer(normalized_reference.replace(' ', ''), normalized_hypothesis.replace(' ', '')),
                'language_token': next((id_to_language[token_id] for token_id in token_ids[:4] if token_id in id_to_language), None),
                'ended_with_eos': token_ids[-1] == processor.tokenizer.eos_token_id,
                'generated_token_ids': token_ids,
            }
            predictions.append(record)
            print(f'\n{precision} | {mode} | {record["clip_id"]} | CMI {record["cmi"]}')
            print('REFERENCE:', reference)
            print('WHISPER preview: ', hypothesis[:240])
            print(f'WER: {record["wer"]:.1%} | CER: {record["cer"]:.1%}')
            if not record['ended_with_eos']:
                print('CHECK: output did not end with EOS; the token limit may have truncated it.')

            del inputs, generated

    del model
    gc.collect()
    torch.cuda.empty_cache()


### 3.2 Compare the Four Conditions

Verify identical clip coverage. Report corpus WER/CER and outputs without EOS; preserve all failures rather than dropping them. A change suggests precision sensitivity in this configuration, not a general claim that one precision is always better. If both remain poor, inspect the reference checks and test implementation equivalence before the full benchmark.


In [ ]:
if len(predictions) != 4 * len(clips):
    raise ValueError('Expected all 20 predictions.')
summary = {}
for precision in ['fp16', 'fp32']:
    for mode in ['auto', 'forced_ne']:
        rows = [row for row in predictions if row['precision'] == precision and row['mode'] == mode]
        if len(rows) != len(clips) or {row['clip_id'] for row in rows} != set(expected_clips):
            raise ValueError(f'Incomplete or duplicate coverage in {precision}/{mode}.')
        refs = [row['normalized_reference'] for row in rows]
        hyps = [row['normalized_hypothesis'] for row in rows]
        summary[precision + '/' + mode] = {
            'clips': len(rows), 'wer': wer(refs, hyps),
            'cer': cer([text.replace(' ', '') for text in refs], [text.replace(' ', '') for text in hyps]),
            'outputs_without_eos': sum(not row['ended_with_eos'] for row in rows),
        }
print(json.dumps(summary, indent=2))
print('Reference checks:', {verdict: sum(row['reference_check'] == verdict for row in listening_checks)
                            for verdict in ['match', 'mismatch', 'unsure']})


### 4.1 Download the Diagnostic ZIP

Send the new ZIP back for analysis. It includes all 20 outputs, your listening notes, original baseline/settings, confirmed manifest, runtime versions, and GPU details. Original pilot results are preserved.


In [ ]:
created_at = datetime.now(timezone.utc)
bundle = {
    'created_at_utc': created_at.isoformat(), 'purpose': 'audio/precision validation diagnostic, not final benchmark',
    'dataset': baseline['dataset'], 'model': baseline['model'], 'original_pilot': baseline,
    'manifest': [clip['identity'] for clip in clips], 'listening_checks': listening_checks,
    'predictions': predictions, 'summary': summary, 'generation_configs': mode_configs,
    'generation_overrides': baseline['generation_overrides'],
    'runtime': {'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
                'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'dtypes': ['fp16', 'fp32'],
                'attention_implementation': 'sdpa', 'allow_tf32_matmul': False},
}
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
environment += '\n\n' + subprocess.check_output(['nvidia-smi'], text=True)
archive_path = Path('whisper_precision_diagnostic_' + created_at.strftime('%Y%m%dT%H%M%S%fZ') + '.zip')
with zipfile.ZipFile(archive_path, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('results.json', json.dumps(bundle, ensure_ascii=False, indent=2, allow_nan=False))
    archive.writestr('environment.txt', environment)
print('Saved:', archive_path.resolve())
from google.colab import files
files.download(str(archive_path))
